In [ ]:
from tokenise import BPE_token
from pathlib import Path
import os

import torch
from transformers import GPT2Config, GPT2LMHeadModel, GPT2Tokenizer
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments
from torch.utils.data import  DataLoader
from transformers import  LineByLineTextDataset

## train a tokenizer

In [ ]:
# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize the tokenizer
tokenizer = BPE_token()

# The 'text' folder contains all the files
paths = [str(x) for x in Path("./text/").glob("**/*.txt")]

# Train the tokenizer
tokenizer.bpe_train(paths)

# Save the tokenized data in the specified folder
save_path = 'tokenized_data'
tokenizer.save_tokenizer(save_path)

## generate corpus

In [ ]:
from right_corpus import NP, NPVP, SOV
print(len(NP + NPVP + SOV))

## evaluate functions

In [ ]:
def evaluate_np_npvp_sov(model, tokenizer, test_dataset, per_for_train,temp):
    np_generated = []
    npvp_generated = []
    sov_generated = []
    model.eval()
    correct_count = 0
    total_generated = 0
    new = 0
    cor = []
    cor_new = []
    
    with open("./text/all_sentences.txt", "r", encoding="utf-8") as f:
        full_corpus = f.read().splitlines()
    with open(f"./text/train{per_for_train}.txt", "r", encoding="utf-8") as ff:
        train_corpus = ff.read().splitlines()
    minus = [x for x in full_corpus if x not in train_corpus]
    for example in test_dataset:
        #acc_mark = 0
        #print(example)
        
        first_token = example["input_ids"][0]
        second_token = example["input_ids"][1]
        
        #start_token = tokenizer.bos_token_id

       
        input_ids = torch.tensor([[0,first_token,second_token]]).to(model.device)
        #input_ids = torch.tensor([[0]]).to(device)
        for gen_len in range(5,20):
            with torch.no_grad():
                #is_complete_sentence = False
                #gen_len = 5
               
                #print("Model device:", model.device)
                generated_ids = model.generate(
                    input_ids,
                    max_length=gen_len,  
                    num_return_sequences=1,
                    pad_token_id=tokenizer.eos_token_id,
                    attention_mask=torch.ones_like(input_ids),
                    return_dict_in_generate=True,
                    temperature=temp,  
                    do_sample=True  
                )

            
            generated_sequence = tokenizer.decode(generated_ids.sequences[0], skip_special_tokens=True)
            #print(input_ids)
            
            new_mark = 0
            
            if generated_sequence in full_corpus:
                #print(input_ids)
                #print(generated_sequence)
                correct_count += 1
                cor.append(generated_sequence)
                #acc_mark = 1
                if generated_sequence in minus:
                    new += 1
                    new_mark = 1
                    cor_new.append(generated_sequence) 
                    #print(generated_sequence)
                if generated_sequence in NP:
                    np_generated.append(generated_sequence)
                if generated_sequence in NPVP:
                    npvp_generated.append(generated_sequence)
                if generated_sequence in SOV:
                    sov_generated.append(generated_sequence)
                break
                
        if new_mark == 0 and generated_sequence in full_corpus:
            generated_ids = model.generate(
                    input_ids,
                    max_length= gen_len + 1, 
                    num_return_sequences=1,
                    pad_token_id=tokenizer.eos_token_id,
                    attention_mask=torch.ones_like(input_ids),
                    return_dict_in_generate=True,
                    temperature=temp, 
                    do_sample=True
                )
            generated_sequence = tokenizer.decode(generated_ids.sequences[0], skip_special_tokens=True)
            if generated_sequence in minus:
                    new += 1
                    new_mark = 1 
                    cor.append(generated_sequence)
                    cor_new.append(generated_sequence)
                    #print(generated_sequence)
            if generated_sequence in NP:
                np_generated.append(generated_sequence)
            if generated_sequence in NPVP:
                npvp_generated.append(generated_sequence)
            if generated_sequence in SOV:
                sov_generated.append(generated_sequence)
        total_generated += 1
        if generated_sequence in full_corpus:
            #print(generated_sequence)
            cor.append(generated_sequence)
        if generated_sequence in minus:
            #print(generated_sequence)
            cor_new.append(generated_sequence)
        if generated_sequence in NP:
            np_generated.append(generated_sequence)
        if generated_sequence in NPVP:
            npvp_generated.append(generated_sequence)
        if generated_sequence in SOV:
            sov_generated.append(generated_sequence)
    ## if acc_mark ==1 !!!!!
    
    #print(generated_sequence)
    new_acc = new / ( new + (total_generated - correct_count) )
    accuracy = correct_count / total_generated
    new_generated = len(set(cor)) / total_generated
    if len(set(cor)) != 0:
        new_no_re = len(set(cor_new)) / len(set(cor))
        np_per = len(set(np_generated)) / len(set(cor))#why not len(NP)
        npvp_per = len(set(npvp_generated)) / len(set(cor))#why not len(NPVP)
        sov_per = len(set(sov_generated)) / len(set(cor))#why not len(SOV)
    else:
        new_no_re = 0
        np_per = 0#why not len(NP)
        npvp_per = 0#why not len(NPVP)
        sov_per = 0#why not len(SOV)

    return (accuracy, new_acc, new_generated, new_no_re, np_per, npvp_per, sov_per)

## hyperparameter

In [ ]:
#hyperparams
layers = 2
#per_for_train = '10'
per_train_list = ['_human']
temp=1
lr = 5e-4
train_epochs = 200
highest_accuracy = 0.91 

evaluation_results = [[] for _ in range(len(per_train_list))]
evaluation_new_results = [[] for _ in range(len(per_train_list))]
evaluation_new_hit_results = [[] for _ in range(len(per_train_list))]
evaluation_new_hit_gg = [[] for _ in range(len(per_train_list))]
evaluation_np_per = [[] for _ in range(len(per_train_list))]
evaluation_npvp_per = [[] for _ in range(len(per_train_list))]
evaluation_sov_per = [[] for _ in range(len(per_train_list))]

## training

In [ ]:
for index,per_for_train in enumerate(per_train_list):
    tokenizer = GPT2Tokenizer.from_pretrained(save_path)
    tokenizer.add_special_tokens({
        "eos_token": "</s>",
        "bos_token": "<s>",
        "unk_token": "<unk>",
        "pad_token": "<pad>",
        "mask_token": "<mask>"
    })


    # Create configuration
    config = GPT2Config(
        vocab_size=tokenizer.vocab_size,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        n_layer= layers ,
    )

    # Create the model and move it to GPU
    model = GPT2LMHeadModel(config).to(device)

    # Set the attention mask and pad token id
    model.config.pad_token_id = tokenizer.eos_token_id

    # Configure DataCollator
    data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

    # Read training data
    train_file = f"./text/train{per_for_train}.txt"
    train_dataset = LineByLineTextDataset(
        tokenizer=tokenizer,
        file_path=train_file,
        block_size=128  # Adjust block_size as per your requirement
    )

    # Create PyTorch DataLoader
    train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)

    # Read test data
    test_file = "./text/all_sentences.txt" 
    test_dataset = LineByLineTextDataset(
        tokenizer=tokenizer,
        file_path=test_file,
        block_size=128  # Adjust block_size as per your requirement
    )

    # Remove the 'device' argument from TrainingArguments
    training_args = TrainingArguments(
        output_dir="./output",  # Path to save the model
        overwrite_output_dir=True,
        num_train_epochs=1,
        per_device_train_batch_size=4,
        save_steps=10_000,  # Save the model checkpoint every X steps
        save_total_limit=2,  # Limit the total number of saved models
        prediction_loss_only=True,  # Only calculate prediction loss and not other losses
        learning_rate=lr,
    )

    # Initialize Trainer without the 'device' argument
    trainer = Trainer(
        model=model,
        args=training_args,
        data_collator=data_collator,
        train_dataset=train_dataset,
    )



    # Train and test
    acc_track = 0
    acc_track1 = 0
    acc_track2 = 0
    for epoch in range(train_epochs):
        trainer.train()
        if epoch != 999:
        #if ( epoch==0 ) or ( epoch==1 ) or ( ((epoch+1) % 5 == 0 ) and epoch > 2 and epoch < 100 )  or  ( epoch > 100 and (epoch+1) % 50 ==0) :
            reresult = evaluate_np_npvp_sov(model, tokenizer, test_dataset,per_for_train,temp)
            accuracy = reresult[0]
            new_rate = reresult[1]
            new_hit = reresult[2]
            new_gp = reresult[3]
            np_per = reresult[4]
            npvp_per = reresult[5]
            sov_per = reresult[6]
            evaluation_results[index].append(accuracy)
            evaluation_new_results[index].append(new_rate)
            evaluation_new_hit_results[index].append(new_hit)
            evaluation_new_hit_gg[index].append(new_gp)
            evaluation_np_per[index].append(np_per)
            evaluation_npvp_per[index].append(npvp_per)
            evaluation_sov_per[index].append(sov_per)
            
            

            output_dir = f"./predict_model/tuned_epoch_{epoch}_new_rate_{new_rate}_acc_{accuracy}_model"
            model.save_pretrained(output_dir)
                

            print(f"Accuracy in {per_for_train}%corpus after {epoch+1} epochs: {accuracy:.2f}")
            print(f"New_rate in {per_for_train}%corpus after {epoch+1} epochs: {new_rate:.2f}")
            print(f"New_generated  in {per_for_train}%corpus after {epoch+1} epochs: {new_hit:.2f}")
            print(f"New_generated_without_repete  in {per_for_train}%corpus after {epoch+1} epochs: {new_gp:.2f}")
            print(f"np_per in {per_for_train}%corpus after {epoch+1} epochs: {np_per:.2f}")
            print(f"npvp_per in {per_for_train}%corpus after {epoch+1} epochs: {npvp_per:.2f}")
            print(f"sov_per in {per_for_train}%corpus after {epoch+1} epochs: {sov_per:.2f}")


## ploting

In [ ]:
new_rates_list = []
pre_models_list = []
pre_models_path = './predict_model'
for name_id in range(200):
    for name in os.listdir(pre_models_path):
        if name[0] == 'f' and name.split('_')[3] == str(name_id):
            #if  (not (float(name.split('_')[6]) in new_rates_list)):
            pre_models_list.append(pre_models_path+name)
            new_rates_list.append(float(name.split('_')[6]))

In [ ]:
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

epochs = range(200)

for i, per_for_train in enumerate(per_train_list):
    plt.figure(figsize=(8, 8))
    
 
    original_new_results = new_rates_list#evaluation_new_results[i]
    smoothed_new_results = gaussian_filter1d(original_new_results, sigma=2)
    

    original_max_y = max(original_new_results)
    smoothed_max_y = max(smoothed_new_results)
    offset = original_max_y - smoothed_max_y
    adjusted_smoothed_new_results = smoothed_new_results + offset  
    
    plt.plot(epochs, adjusted_smoothed_new_results, '#075B72', label='New Rate', linewidth=5)
    
    max_y = max(adjusted_smoothed_new_results)  
    max_x = epochs[adjusted_smoothed_new_results.tolist().index(max_y)] 
    plt.axhline(y=max_y, color='#075B72', linestyle='--', linewidth=1.5, alpha=0.8)  
    plt.text(50, max_y + 0.02, f'Max: {max_y:.2f}', fontsize=25, color='#075B72')  
    
   
    plt.ylim(0, 1.0)  
    plt.xticks(fontsize=25)
    plt.yticks(fontsize=25)
    plt.xlabel('Epochs', fontsize=25)
    plt.ylabel('Accuracy', fontsize=25)
    plt.grid(True)
    plt.legend(loc='lower right', fontsize=20)
    plt.tight_layout()
   
    plt.show()
